# SpaCy Term Extraction

This notebook demonstrates two approaches to term extraction using SpaCy:
1. **Baseline**: Rule-based extraction with EntityRuler
2. **Trained**: Custom NER model fine-tuned for term extraction

## Setup and Imports
For complete information about the available SpaCy models: https://spacy.io/models/en

In [1]:
!pip install spacy-lookups-data
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_md
!python -m spacy download en_core_web_lg

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\marti\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------------ --------------------------- 3.9/12.8 MB 26.0 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 31.2 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 28.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ------ --------------------------------- 5.2/33.5 MB 29.0 MB/s eta 0:00:01
     ------------- ------------------------- 11.3/33.5 MB 29.4 MB/s eta 0:00:01
     -------------------- ------------------ 17.8/33.5 MB 31.2 MB/s eta 0:00:01
     ---------------------------- ---------- 24.4/33.5 MB 31.5 MB/s eta 0:00:01
     ------------------------------------ -- 30.9/33.5 MB 31.1 MB/s eta 0:00:01
     ---------------------------------------- 33.5/33.5 MB 29.9 MB/s  0:00:01
✔ Download and installation successful


In [3]:
import json
import os
import random
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np

import spacy
from spacy.tokens import DocBin, Doc
from spacy.training import Example
from spacy.util import minibatch, compounding
from spacy.pipeline import EntityRuler
from tqdm import tqdm

# Load English model
try:
    nlp = spacy.load('en_core_web_sm')
    print("English model loaded successfully")
except:
    print("Model not found. Install with: python -m spacy download en_core_web_sm")

English model loaded successfully


In [4]:
random.seed(42)
np.random.seed(42)

## Data Loading and Processing

In [ ]:
# Load training and test data
train_data = json.load(open('../data/taskA/train.json', 'r', encoding='utf-8'))
test_data = json.load(open('../data/taskA/test.json', 'r', encoding='utf-8'))

print(f"Training documents: {len(train_data)}")
print(f"Test documents: {len(test_data)}")

print(f"\nExample document:")
for id, content in train_data.items():
    print(f"Document ID: {id}")
    print(f"  Text: {content['text']}")
    print(f"  Terms: {content['terms']}")
    break  

Training documents: 540
Test documents: 567

Example document:
Document ID: mh_125
  Text: The role of gut microbiota in the pathogenesis and treatment of postpartum depression.  Postpartum depression (PPD) is a common complication of pregnancy in women, and its pathogenesis mainly involves disturbances of the neuroendocrine regulation, immune system, neurotransmitters, hormone secretion, and the gut microbiome. Gut microbes play essential physiological and pathological roles in the gut-brain axis' pathways which are involved in various central nervous system (CNS) and psychiatric disorders, including PPD. Numerous studies have identified the fundamental role of the gut-brain axis in the pathogenesis and treatment of PPD patients and also correlates with other pathogenic mechanisms of PPD. Disturbances in gut microbes are associated with the disruption of multiple signaling pathways and systems that ultimately lead to PPD development. This review aimed to elucidate the potential connec

## Evaluation Metrics

Using the official evaluation metrics from the competition.

In [6]:
def micro_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Precision, Recall, and F1 score 
    based on individual term matching (micro-average).
    
    Args:
        gold_standard: List of lists, where each inner list contains gold standard terms
        system_output: List of lists, where each inner list contains extracted terms
    
    Returns:
        Tuple containing (precision, recall, f1, tp, fp, fn)
    """
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0
    
    # Iterate through each item's gold standard and system output terms
    for gold, system in zip(gold_standard, system_output):
        # Convert to sets for efficient comparison
        gold_set = set(gold)
        system_set = set(system)
        
        # Calculate TP, FP, FN for the current item
        true_positives = len(gold_set.intersection(system_set))
        false_positives = len(system_set - gold_set)
        false_negatives = len(gold_set - system_set)
        
        # Accumulate totals across all items
        total_true_positives += true_positives
        total_false_positives += false_positives
        total_false_negatives += false_negatives
    
    # Calculate Precision, Recall, and F1 score (micro-average)
    precision = total_true_positives / (total_true_positives + total_false_positives) if (total_true_positives + total_false_positives) > 0 else 0
    recall = total_true_positives / (total_true_positives + total_false_negatives) if (total_true_positives + total_false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1, total_true_positives, total_false_positives, total_false_negatives


def type_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Type Precision, Type Recall, and Type F1 score
    based on the set of unique terms extracted at least once across the entire dataset.
    
    Args:
        gold_standard: List of lists, where each inner list contains gold standard terms
        system_output: List of lists, where each inner list contains extracted terms
    
    Returns:
        Tuple containing (type_precision, type_recall, type_f1)
    """
    # Get the set of all unique gold standard terms across the dataset
    all_gold_terms = set()
    for item_terms in gold_standard:
        all_gold_terms.update(item_terms)
    
    # Get the set of all unique system extracted terms across the dataset
    all_system_terms = set()
    for item_terms in system_output:
        all_system_terms.update(item_terms)
    
    # Calculate True Positives (terms present in both sets)
    type_true_positives = len(all_gold_terms.intersection(all_system_terms))
    
    # Calculate False Positives (terms in system output but not in gold standard)
    type_false_positives = len(all_system_terms - all_gold_terms)
    
    # Calculate False Negatives (terms in gold standard but not in system output)
    type_false_negatives = len(all_gold_terms - all_system_terms)
    
    # Calculate Type Precision, Type Recall, and Type F1 score
    type_precision = type_true_positives / (type_true_positives + type_false_positives) if (type_true_positives + type_false_positives) > 0 else 0
    type_recall = type_true_positives / (type_true_positives + type_false_negatives) if (type_true_positives + type_false_negatives) > 0 else 0
    type_f1 = 2 * (type_precision * type_recall) / (type_precision + type_recall) if (type_precision + type_recall) > 0 else 0
    
    return type_precision, type_recall, type_f1


# Test: Simple case
gold_test = [['term1', 'term2'], ['term3']]
pred_test = [['term1', 'term4'], ['term3']]
precision, recall, f1, tp, fp, fn = micro_f1_score(gold_test, pred_test)
assert tp == 2  # term1 and term3
assert fp == 1  # term4
assert fn == 1  # term2
print("Evaluation functions work correctly")
print(f"  Test metrics: P={precision:.2f}, R={recall:.2f}, F1={f1:.2f}")

# Test type-level metrics
type_p, type_r, type_f1 = type_f1_score(gold_test, pred_test)
print(f"  Type metrics: P={type_p:.2f}, R={type_r:.2f}, F1={type_f1:.2f}")

Evaluation functions work correctly
  Test metrics: P=0.67, R=0.67, F1=0.67
  Type metrics: P=0.67, R=0.67, F1=0.67


## Baseline Model: Rule-Based EntityRuler

Simple approach using SpaCy's EntityRuler:
- Creates exact match patterns from training terms
- Fast and deterministic
- No generalization to unseen terms

In [7]:
class SpacyRuleBaseline:
    """Rule-based extractor using EntityRuler."""
    
    def __init__(self, model: str = 'en_core_web_sm'):
        try:
            self.nlp = spacy.load(model)
        except:
            self.nlp = spacy.blank('it')
        self.ruler = None
    
    def build(self, terms: List[str]):
        """Build EntityRuler patterns from term list."""
        # Create patterns (sorted by length to prioritize longer matches)
        patterns = []
        for t in sorted(set(terms), key=len, reverse=True):
            if t:
                patterns.append({'label': 'TERM', 'pattern': t})
        
        # Remove existing entity_ruler if present
        if 'entity_ruler' in self.nlp.pipe_names:
            self.nlp.remove_pipe('entity_ruler')
        
        # Add new EntityRuler
        self.nlp.add_pipe('entity_ruler', name='entity_ruler', first=True)
        self.ruler = self.nlp.get_pipe('entity_ruler')
        self.ruler.add_patterns(patterns)
        
        print(f"Built EntityRuler with {len(patterns)} patterns")
    
    def predict(self, texts: List[str]) -> List[List[str]]:
        """Extract terms from texts."""
        if self.ruler is None:
            raise RuntimeError("Model not built. Call build() first.")
        
        results = []
        for doc in tqdm(self.nlp.pipe(texts, batch_size=32), desc="Predicting", total=len(texts)):
            terms = [ent.text for ent in doc.ents if ent.label_ == 'TERM']
            results.append(terms)
        return results
    
    def save(self, path: str):
        """Save EntityRuler patterns."""
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        if self.ruler:
            self.ruler.to_disk(path)
            print(f"Model saved to {path}")
    
    def load(self, path: str):
        """Load EntityRuler patterns."""
        if 'entity_ruler' in self.nlp.pipe_names:
            self.nlp.remove_pipe('entity_ruler')
        
        self.nlp.add_pipe('entity_ruler', name='entity_ruler', first=True)
        self.ruler = self.nlp.get_pipe('entity_ruler')
        self.ruler.from_disk(path)
        print(f"Model loaded from {path}")


# Test: Simple predictions
baseline = SpacyRuleBaseline()
baseline.build(['Gut microbiota disturbances', 'adults'])
test_preds = baseline.predict(['Gut microbiota disturbances in hospitalized older adults with malnutrition and clinical outcomes.'])
assert 'Gut microbiota disturbances' in test_preds[0]
assert 'adults' in test_preds[0]
print("Baseline model works correctly")
print(f"  Test predictions: {test_preds[0]}")

Built EntityRuler with 2 patterns


Predicting: 100%|██████████| 1/1 [00:00<00:00, 42.52it/s]

Baseline model works correctly
  Test predictions: ['Gut microbiota disturbances', 'adults']


### Run and Evaluate Baseline Model

In [8]:
# Extract unique terms from training data
train_terms = set()
for id, content in train_data.items():
    train_terms.update(t for t in content['terms'] if t)

print(f"Unique training terms: {len(train_terms)}")

Unique training terms: 6014


In [9]:
# Build baseline model
baseline_model = SpacyRuleBaseline()
baseline_model.build(sorted(train_terms))

# Predict on test set
test_texts = [content['text'] for id, content in test_data.items()]
test_gold = [content['terms'] for id, content in test_data.items()]

print("\nRunning baseline predictions...")
baseline_preds = baseline_model.predict(test_texts)

# Evaluate
precision, recall, f1, tp, fp, fn = micro_f1_score(test_gold, baseline_preds)
type_precision, type_recall, type_f1 = type_f1_score(test_gold, baseline_preds)

print("\nBaseline Model Results:")
print("Micro-averaged metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  TP={tp}, FP={fp}, FN={fn}")
print("\nType-level metrics:")
print(f"  Type Precision: {type_precision:.4f}")
print(f"  Type Recall:    {type_recall:.4f}")
print(f"  Type F1 Score:  {type_f1:.4f}")

# Store metrics for later comparison
baseline_metrics = {
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'type_precision': type_precision,
    'type_recall': type_recall,
    'type_f1': type_f1
}

Built EntityRuler with 6014 patterns

Running baseline predictions...


Predicting: 100%|██████████| 567/567 [00:18<00:00, 30.97it/s]


Baseline Model Results:
Micro-averaged metrics:
  Precision: 0.5183
  Recall:    0.6035
  F1 Score:  0.5576
  TP=11905, FP=11066, FN=7823

Type-level metrics:
  Type Precision: 0.8352
  Type Recall:    0.2578
  Type F1 Score:  0.3940


In [10]:
# Save baseline model
baseline_model.save('models/spacy_baseline')

Model saved to models/spacy_baseline


## Trained Model: Custom NER

Neural approach that learns from examples:
- Fine-tunes SpaCy's NER model on term extraction task
- Learns patterns and context from labeled data
- Can generalize to similar terms not seen during training

In [ ]:
class SpacyTrainedModel:
    """Trainable NER model for term extraction."""
    
    def __init__(self, model: str = 'en_core_web_sm'):
        self.model_name = model
        self.nlp = None
    
    def _prepare_training_data(self, sentences: List[str], term_lists: List[List[str]]) -> List[Example]:
        """Convert to SpaCy training format with character-span annotations."""
        training_data = []
        
        for sent_text, terms in zip(sentences, term_lists):
            doc = self.nlp.make_doc(sent_text)
            entities = []
            
            # Find character spans for each term
            for term in terms:
                if not term:
                    continue
                
                # Find all occurrences
                start_idx = 0
                while True:
                    start_idx = sent_text.find(term, start_idx)
                    if start_idx == -1:
                        break
                    
                    end_idx = start_idx + len(term)
                    span = doc.char_span(start_idx, end_idx, label='TERM', alignment_mode='expand')
                    if span is not None:
                        entities.append((span.start_char, span.end_char, 'TERM'))
                    
                    start_idx = end_idx
            
            # Remove overlapping entities
            # It might be that overlapping occurrences exist and should be captured
            # For example, if "gut microbiota composition" is found, we might want to also capture "gut microbiota" 
            # In this case, we need to ensure that both spans are included
            entities = self._remove_overlapping(entities)
            example = Example.from_dict(doc, {'entities': entities})
            training_data.append(example)
        
        return training_data
    
    def _remove_overlapping(self, entities: List[Tuple[int, int, str]]) -> List[Tuple[int, int, str]]:
        """Keep longer spans when entities overlap."""
        if not entities:
            return []
        
        # Sort by start, then by length (descending)
        entities = sorted(entities, key=lambda x: (x[0], -(x[1] - x[0])))
        
        non_overlapping = []
        for start, end, label in entities:
            overlaps = False
            for prev_start, prev_end, _ in non_overlapping:
                if not (end <= prev_start or start >= prev_end):
                    overlaps = True
                    break
            if not overlaps:
                non_overlapping.append((start, end, label))
        
        return non_overlapping
    
    def train(self, sentences: List[str], term_lists: List[List[str]], 
              n_iter: int = 30, dropout: float = 0.2, batch_size: int = 8):
        """Train NER model on labeled data."""
        print(f"Initializing model: {self.model_name}")
        
        # Load base model
        try:
            self.nlp = spacy.load(self.model_name)
            pretrained_ner = "ner" in self.nlp.pipe_names
        except OSError:
            print("Model not found; using a blank English model")
            self.nlp = spacy.blank("en")
            pretrained_ner = False
        
        # Setup NER
        if 'ner' not in self.nlp.pipe_names:
            ner = self.nlp.add_pipe('ner')
        else:
            ner = self.nlp.get_pipe('ner')
        ner.add_label('TERM')
        
        # Prepare training data
        print("Preparing training examples...")
        train_examples = self._prepare_training_data(sentences, term_lists)
        train_examples = [ex for ex in train_examples if len(ex.reference.ents) > 0] # Keep only examples with entities
        print(f"Training on {len(train_examples)} examples")
        
        # Train
        if not train_examples:
            raise ValueError("No valid training examples containing terms were found.")

        other_pipes = [pipe for pipe in self.nlp.pipe_names if pipe != "ner"]

        with self.nlp.disable_pipes(*other_pipes):
            if pretrained_ner:
                optimizer = self.nlp.resume_training()
            else:
                optimizer = self.nlp.initialize(
                    get_examples=lambda: train_examples
                )

            for iteration in tqdm(range(n_iter), desc="Training"):
                random.shuffle(train_examples)
                losses = {}

                batches = minibatch(
                    train_examples,
                    size=compounding(4.0, batch_size, 1.001),
                )

                for batch in batches:
                    self.nlp.update(
                        batch,
                        sgd=optimizer,
                        drop=dropout,
                        losses=losses,
                    )

                if iteration % 5 == 0:
                    print(
                        f"  Iteration {iteration}: "
                        f"Loss = {losses.get('ner', 0):.3f}"
                    )

        print("Training complete!")
    
    def predict(self, sentences: List[str]) -> List[List[str]]:
        """Extract terms from sentences."""
        if self.nlp is None:
            raise RuntimeError("Model not trained. Call train() or load() first.")
        
        results = []
        for doc in self.nlp.pipe(sentences, batch_size=32):
            terms = [ent.text for ent in doc.ents if ent.label_ == 'TERM']
            results.append(terms)
        return results
    
    def save(self, path: str):
        """Save trained model."""
        if self.nlp is None:
            raise RuntimeError("No model to save")
        
        output_dir = Path(path)
        output_dir.mkdir(parents=True, exist_ok=True)
        self.nlp.to_disk(output_dir)
        print(f"Model saved to {output_dir}")
    
    def load(self, path: str):
        """Load trained model."""
        self.nlp = spacy.load(path)
        if 'ner' not in self.nlp.pipe_names:
            raise ValueError("Loaded model doesn't have NER component")
        print(f"Model loaded from {path}")


# Test: Simple training
test_model = SpacyTrainedModel()
test_sents = [
    "Association between gut microbiota dysbiosis and poor functional outcomes in acute ischemic stroke patients with COVID-19 infection.", 
    "Gut microbiota disturbances in hospitalized older patients with malnutrition and clinical outcomes."
]
test_terms = [
    ["gut microbiota dysbiosis", "patients"], 
    ["Gut microbiota disturbances", "patients"]
]
test_model.train(test_sents, test_terms, n_iter=100, dropout=0.0, batch_size=2,)
test_pred = test_model.predict(["Integrated multi-omics analysis reveals gut microbiota dysbiosis and systemic disturbance in major depressive disorder patients."])
assert 'gut microbiota dysbiosis' in test_pred[0] and 'patients' in test_pred[0]
print(f"✓ Trained model works correctly")
print(f"  Test prediction: {test_pred[0]}")

Initializing model: en_core_web_sm
Preparing training examples...
Training on 2 examples


Training:   0%|          | 0/100 [00:00<?, ?it/s]

  Iteration 0: Loss = 8.717


Training:   6%|▌         | 6/100 [00:00<00:03, 28.32it/s]

  Iteration 5: Loss = 5.661


Training:  10%|█         | 10/100 [00:00<00:02, 30.77it/s]

... (31 outputs left out)

Training:  94%|█████████▍| 94/100 [00:02<00:00, 32.80it/s]

  Iteration 95: Loss = 0.000


Training: 100%|██████████| 100/100 [00:03<00:00, 31.03it/s]

Training complete!
✓ Trained model works correctly
  Test prediction: ['gut microbiota dysbiosis', 'patients']


### Train and Evaluate Trained Model

Note: This cell might take several minutes to run.

**Additional configurations to test**
- Keep overlapping entities
- Keep documents with 0 entities in the training set
- Change hyperparameters (*n_iter*, *dropout*, *batch_size*)

In [12]:
# Prepare training data
train_texts = [content['text'] for content in train_data.values()]
train_term_lists = [content['terms'] for content in train_data.values()]

# Initialize and train model
trained_model = SpacyTrainedModel(model='en_core_web_sm')

trained_model.train(
    train_texts, 
    train_term_lists,
    n_iter=100,
    dropout=0.2,
    batch_size=8
)

Initializing model: en_core_web_sm
Preparing training examples...
Training on 540 examples


Training:   1%|          | 1/100 [00:20<33:14, 20.15s/it]

  Iteration 0: Loss = 28328.018


Training:   6%|▌         | 6/100 [01:57<30:27, 19.44s/it]

  Iteration 5: Loss = 9208.549


Training:  11%|█         | 11/100 [03:35<29:15, 19.72s/it]

... (33 outputs left out)

Training:  96%|█████████▌| 96/100 [47:15<01:28, 22.16s/it]

  Iteration 95: Loss = 1371.318


Training: 100%|██████████| 100/100 [48:35<00:00, 29.15s/it]

Training complete!


In [13]:
# Predict on test set
print("Running trained model predictions...")
trained_preds = trained_model.predict(test_texts)

# Evaluate
precision, recall, f1, tp, fp, fn = micro_f1_score(test_gold, trained_preds)
type_precision, type_recall, type_f1 = type_f1_score(test_gold, trained_preds)

print("\nTrained Model Results:")
print("Micro-averaged metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  TP={tp}, FP={fp}, FN={fn}")
print("\nType-level metrics:")
print(f"  Type Precision: {type_precision:.4f}")
print(f"  Type Recall:    {type_recall:.4f}")
print(f"  Type F1 Score:  {type_f1:.4f}")

# Store metrics for later comparison
trained_metrics = {
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'type_precision': type_precision,
    'type_recall': type_recall,
    'type_f1': type_f1
}

Running trained model predictions...

Trained Model Results:
Micro-averaged metrics:
  Precision: 0.7941
  Recall:    0.6956
  F1 Score:  0.7416
  TP=13723, FP=3559, FN=6005

Type-level metrics:
  Type Precision: 0.7192
  Type Recall:    0.5519
  Type F1 Score:  0.6245


In [14]:
# Save trained model
trained_model.save('models/spacy_trained')

Model saved to models\spacy_trained


## Results Comparison

In [15]:
import pandas as pd

# Micro-averaged comparison
results_df = pd.DataFrame([
    {
        'Model': 'Baseline (Rules)',
        'Precision': baseline_metrics['precision'],
        'Recall': baseline_metrics['recall'],
        'F1': baseline_metrics['f1']
    },
    {
        'Model': 'Trained (NER)',
        'Precision': trained_metrics['precision'],
        'Recall': trained_metrics['recall'],
        'F1': trained_metrics['f1']
    }
])

print("Micro-averaged Metrics:")
print(results_df.to_markdown(index=False))

# Type-level comparison
type_results_df = pd.DataFrame([
    {
        'Model': 'Baseline (Rules)',
        'Type Precision': baseline_metrics['type_precision'],
        'Type Recall': baseline_metrics['type_recall'],
        'Type F1': baseline_metrics['type_f1']
    },
    {
        'Model': 'Trained (NER)',
        'Type Precision': trained_metrics['type_precision'],
        'Type Recall': trained_metrics['type_recall'],
        'Type F1': trained_metrics['type_f1']
    }
])

print("\n\nType-level Metrics:")
print(type_results_df.to_markdown(index=False))

# Show improvement
f1_improvement = (trained_metrics['f1'] - baseline_metrics['f1']) / baseline_metrics['f1'] * 100
type_f1_improvement = (trained_metrics['type_f1'] - baseline_metrics['type_f1']) / baseline_metrics['type_f1'] * 100
print(f"\n\nMicro F1 Score improvement: {f1_improvement:+.1f}%")
print(f"Type F1 Score improvement: {type_f1_improvement:+.1f}%")

Micro-averaged Metrics:
| Model            |   Precision |   Recall |       F1 |
|:-----------------|------------:|---------:|---------:|
| Baseline (Rules) |    0.518262 | 0.603457 | 0.557624 |
| Trained (NER)    |    0.794063 | 0.69561  | 0.741583 |


Type-level Metrics:
| Model            |   Type Precision |   Type Recall |   Type F1 |
|:-----------------|-----------------:|--------------:|----------:|
| Baseline (Rules) |         0.835227 |      0.257804 |  0.393996 |
| Trained (NER)    |         0.719183 |      0.551853 |  0.624504 |


Micro F1 Score improvement: +33.0%
Type F1 Score improvement: +58.5%


## Save Predictions to Files

In [17]:
def save_predictions(predictions, 
                     test_data, 
                     output_path: str):
    """Save predictions in competition format."""
    output = {}
    for id, content in test_data.items():
        output[id] = {
            'text': content['text'],
            'terms': predictions[list(test_data.keys()).index(id)]
        }
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(predictions)} predictions to {output_path}")


# Save both sets of predictions
save_predictions(baseline_preds, test_data, 'predictions/subtaskA_spacy_baseline_preds.json')
save_predictions(trained_preds, test_data, 'predictions/subtaskA_spacy_trained_preds.json')

Saved 567 predictions to predictions/subtaskA_spacy_baseline_preds.json
Saved 567 predictions to predictions/subtaskA_spacy_trained_preds.json


## Load and Test Saved Models

In [19]:
# Test loading baseline
loaded_baseline = SpacyRuleBaseline()
loaded_baseline.load('models/spacy_baseline')
test_preds_baseline = loaded_baseline.predict([test_data['mh_50']['text']])
assert test_preds_baseline[0] == baseline_preds[0]
print("Baseline model saved and loaded correctly")

# Test loading trained model
loaded_trained = SpacyTrainedModel()
loaded_trained.load('models/spacy_trained')
test_preds_trained = loaded_trained.predict([test_data['mh_50']['text']])
assert test_preds_trained[0] == trained_preds[0]
print("Trained model saved and loaded correctly")

print("\nAll models successfully saved and can be reloaded!")

Model loaded from models/spacy_baseline


Predicting: 100%|██████████| 1/1 [00:00<00:00, 19.59it/s]

Baseline model saved and loaded correctly


Model loaded from models/spacy_trained
Trained model saved and loaded correctly

All models successfully saved and can be reloaded!


## Example Predictions

In [29]:
# Show example predictions from both models
example_idx = 'mh_50'
example_text = test_data[example_idx]['text']
example_gold_terms = test_data[example_idx]['terms']

baseline_preds = loaded_baseline.predict([example_text])
trained_preds = loaded_trained.predict([example_text])

print(f"Document: {example_text}\n")
print(f"Gold terms: {example_gold_terms}\n")
print(f"Baseline predictions: {baseline_preds[0]}")
print(f"Trained predictions: {trained_preds[0]}\n")

baseline_correct = set(baseline_preds[0]) & set(example_gold_terms)
trained_correct = set(trained_preds[0]) & set(example_gold_terms)

print(f"Baseline correct {len(baseline_correct)/len(example_gold_terms)*100:.2f}%: {sorted(baseline_correct)}")
print(f"Trained correct {len(trained_correct)/len(example_gold_terms)*100:.2f}%: {sorted(trained_correct)}")


Predicting: 100%|██████████| 1/1 [00:00<00:00, 28.90it/s]

Document: Gut-Brain Axis Impact on Canine Anxiety Disorders: New Challenges for Behavioral Veterinary Medicine.  Anxiety disorders in dogs are ever-growing and represent an important concern in the veterinary behavior field. These disorders are often disregarded in veterinary clinical practice, negatively impacting the animal's and owner's quality of life. Moreover, these anxiety disorders can potentially result in the abandonment or euthanasia of dogs. Growing evidence shows that the gut microbiota is a central player in the gut-brain axis. A variety of microorganisms inhabit the intestines of dogs, which are essential in maintaining intestinal homeostasis. These microbes can impact mental health through several mechanisms, including metabolic, neural, endocrine, and immune-mediated pathways. The disruption of a balanced composition of resident commensal communities, or dysbiosis, is implicated in several pathological conditions, including mental disorders such as anxiety. Studies car